<h1 style="color:#2E86AB; font-family:Georgia; text-align:center; border-bottom: 3px solid #2E86AB; padding-bottom:10px;">
🔗 Merge y Feature Engineering
</h1>

**Autor:** Eddy Luis  
**Fecha:** Mayo 2026  
**Herramientas:** Python · Pandas · NumPy

---

## 📌 Descripción
Unimos el dataset de **centros educativos** con los **indicadores de abandono, promoción y reprobación** extraídos de los PDFs del MINERD. Luego creamos las variables (features) que alimentarán el modelo predictivo de deserción escolar.

---

## 📋 Contenido de este Notebook

1. 📦 Importar Librerías y Cargar Datos
2. 🧹 Armonizar Nombres de Distritos
3. 🔗 Merge: Centros + Indicadores
4. 🏷️ Crear Flags de Niveles Ofrecidos
5. 📐 Asignar Tasa de Abandono Relevante
6. 📊 Features a Nivel de Distrito
7. 📈 Tendencia del Abandono (2021→2025)
8. ⚠️ Clasificación de Riesgo
9. 💾 Guardar Dataset Final
10. 🔍 Resumen del Dataset

---

<h2 style="color:#A23B72; font-family:Georgia;">
📦 1. Importar Librerías y Cargar Datos
</h2>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df_centros = pd.read_csv(PROCESSED_DIR / "centros_educativos_limpios.csv")
df_indicadores = pd.read_csv(PROCESSED_DIR / "indicadores_educativos_ancho.csv")

print(f"Centros educativos: {df_centros.shape}")
print(f"Indicadores: {df_indicadores.shape}")

Centros educativos: (18114, 12)
Indicadores: (564, 14)


<h2 style="color:#A23B72; font-family:Georgia;">
🧹 2. Armonizar Nombres de Distritos
</h2>

Hay 3 distritos con nombres diferentes entre ambos datasets:
- `VILLA  ALTAGRACIA` (doble espacio) vs `VILLA ALTAGRACIA`
- `PERALVILLO` vs `ESPERALVILLO`
- `NEIBA` vs `NEYBA`

In [2]:
DISTRITO_MAP = {
    "0404 - VILLA  ALTAGRACIA": "0404 - VILLA ALTAGRACIA",
    "1705 - PERALVILLO": "1705 - ESPERALVILLO",
    "1801 - NEIBA": "1801 - NEYBA",
}

df_centros["distrito"] = df_centros["distrito"].replace(DISTRITO_MAP)

# Verificar
distritos_c = set(df_centros["distrito"].unique())
distritos_i = set(df_indicadores[df_indicadores["tipo"] == "distrito"]["regional_distrito"].unique())
print(f"Distritos centros: {len(distritos_c)}")
print(f"Distritos indicadores: {len(distritos_i)}")
print(f"Intersección: {len(distritos_c & distritos_i)}")
print(f"Sin match: {distritos_c.symmetric_difference(distritos_i)}")

Distritos centros: 122
Distritos indicadores: 122
Intersección: 122
Sin match: set()


<h2 style="color:#A23B72; font-family:Georgia;">
🔗 3. Merge: Centros + Indicadores
</h2>

Cruzamos a nivel **distrito** (más granular que regional). Usamos los indicadores del último año disponible (2024-2025) como referencia principal, con datos históricos para calcular tendencias.

In [3]:
COLS_INDICADOR = [
    "abandono_inicial", "promovido_inicial", "reprobado_inicial",
    "abandono_primario", "promovido_primario", "reprobado_primario",
    "abandono_secundario", "promovido_secundario", "reprobado_secundario",
]

# Indicadores a nivel distrito, último año
año_col = [c for c in df_indicadores.columns if "lectivo" in c][0]
ind_distrito_actual = df_indicadores[
    (df_indicadores["tipo"] == "distrito") &
    (df_indicadores[año_col] == "2024-2025")
][["regional_distrito"] + COLS_INDICADOR].copy()

ind_distrito_actual = ind_distrito_actual.rename(columns={"regional_distrito": "distrito"})

# Merge
df = df_centros.merge(ind_distrito_actual, on="distrito", how="left")

print(f"Shape después del merge: {df.shape}")
print(f"Centros con indicadores: {df['abandono_secundario'].notna().sum()} / {len(df)}")
print(f"Centros sin indicadores: {df['abandono_secundario'].isna().sum()}")
df.head()

Shape después del merge: (18114, 21)
Centros con indicadores: 18114 / 18114
Centros sin indicadores: 0


,regional,distrito,centros,sector,nivel,coordenadas latitud,coordenadas longitud,matricula,planta fisica,provincia,...,año,abandono_inicial,promovido_inicial,reprobado_inicial,abandono_primario,promovido_primario,reprobado_primario,abandono_secundario,promovido_secundario,reprobado_secundario
0,01 - BARAHONA,0101 - PEDERNALES,02334 - HERNANDO GORJON,PUBLICO,INICIAL - PRIMARIO - SECUNDARIO,18.032553,-71.746648,710,16000218 - HERNANDO GORJON,PEDERNALES,...,20222023,1.4,98.6,0.0,5.7,87.7,6.6,7.1,85.7,7.2
1,01 - BARAHONA,0101 - PEDERNALES,02335 - BIENVENIDO MORILLO,PUBLICO,INICIAL - PRIMARIO,18.045131,-71.743074,523,16000416 - BIENVENIDO MORILLO,PEDERNALES,...,20222023,1.4,98.6,0.0,5.7,87.7,6.6,7.1,85.7,7.2
2,01 - BARAHONA,0101 - PEDERNALES,02336 - EZEQUIEL HERNANDEZ MENDEZ,PUBLICO,INICIAL - PRIMARIO - SECUNDARIO,18.040663,-71.748256,652,16000812 - EZEQUIEL HERNANDEZ MENDEZ,PEDERNALES,...,20222023,1.4,98.6,0.0,5.7,87.7,6.6,7.1,85.7,7.2
3,01 - BARAHONA,0101 - PEDERNALES,02337 - EULALIO EMILIO MANCEBO REYES,PUBLICO,INICIAL - PRIMARIO,18.086838,-71.665969,76,16001015 - EULALIO EMILIO MANCEBO REYES,PEDERNALES,...,20222023,1.4,98.6,0.0,5.7,87.7,6.6,7.1,85.7,7.2
4,01 - BARAHONA,0101 - PEDERNALES,02338 - ARSENIO PEREZ PEÑA,PUBLICO,INICIAL - PRIMARIO,18.173358,-71.742050,60,16001119 - ARSENIO PEREZ PEÑA,PEDERNALES,...,20222023,1.4,98.6,0.0,5.7,87.7,6.6,7.1,85.7,7.2


<h2 style="color:#A23B72; font-family:Georgia;">
🏷️ 4. Crear Flags de Niveles Ofrecidos
</h2>

Cada centro puede ofrecer uno o varios niveles (ej. `INICIAL - PRIMARIO - SECUNDARIO`). Creamos columnas binarias para indicar qué niveles ofrece.

In [4]:
df["tiene_inicial"] = df["nivel"].str.contains("INICIAL").astype(int)
df["tiene_primario"] = df["nivel"].str.contains("PRIMARIO").astype(int)
df["tiene_secundario"] = df["nivel"].str.contains("SECUNDARIO").astype(int)
df["tiene_adultos"] = df["nivel"].str.contains("ADULTOS|PREPARA").astype(int)
df["num_niveles"] = df["tiene_inicial"] + df["tiene_primario"] + df["tiene_secundario"] + df["tiene_adultos"]

# Sector como variable numérica
df["es_publico"] = (df["sector"] == "PUBLICO").astype(int)

print("Distribución de niveles ofrecidos:")
print(df[["tiene_inicial", "tiene_primario", "tiene_secundario", "tiene_adultos"]].sum())
print(f"\nSector público: {df['es_publico'].sum()} / {len(df)}")

Distribución de niveles ofrecidos:
tiene_inicial       12130
tiene_primario      12517
tiene_secundario     6216
tiene_adultos        2306
dtype: int64

Sector público: 15275 / 18114


<h2 style="color:#A23B72; font-family:Georgia;">
📐 5. Asignar Tasa de Abandono Relevante
</h2>

Cada centro recibe la tasa de abandono máxima entre los niveles que ofrece. Un centro que ofrece Primario y Secundario toma el mayor abandono de esos dos niveles como indicador de riesgo.

In [5]:
def abandono_relevante(row):
    tasas = []
    if row["tiene_inicial"] and pd.notna(row["abandono_inicial"]):
        tasas.append(row["abandono_inicial"])
    if row["tiene_primario"] and pd.notna(row["abandono_primario"]):
        tasas.append(row["abandono_primario"])
    if row["tiene_secundario"] and pd.notna(row["abandono_secundario"]):
        tasas.append(row["abandono_secundario"])
    return max(tasas) if tasas else np.nan

df["abandono_max"] = df.apply(abandono_relevante, axis=1)
df["abandono_promedio"] = df[["abandono_inicial", "abandono_primario", "abandono_secundario"]].mean(axis=1)

print(f"abandono_max:\n{df['abandono_max'].describe().round(2)}")
print(f"\nabandono_promedio:\n{df['abandono_promedio'].describe().round(2)}")

abandono_max:
count    15808.00
mean         3.79
std          2.20
min          0.00
25%          2.20
50%          3.20
75%          5.50
max         12.60
Name: abandono_max, dtype: float64

abandono_promedio:
count    18114.00
mean         3.25
std          0.99
min          1.33
25%          2.47
50%          3.17
75%          3.67
max          7.40
Name: abandono_promedio, dtype: float64


<h2 style="color:#A23B72; font-family:Georgia;">
📊 6. Features a Nivel de Distrito
</h2>

Calculamos estadísticas agregadas por distrito que caracterizan el contexto educativo local.

In [6]:
distrito_stats = df.groupby("distrito").agg(
    num_centros=("centros", "count"),
    matricula_total=("matricula", "sum"),
    matricula_promedio=("matricula", "mean"),
    matricula_mediana=("matricula", "median"),
    pct_publico=("es_publico", "mean"),
).round(2)

df = df.merge(distrito_stats, on="distrito", how="left")

print(f"Features de distrito agregadas:")
print(distrito_stats.describe().round(2))

Features de distrito agregadas:
       num_centros  matricula_total  matricula_promedio  matricula_mediana  \
count       122.00           122.00              122.00             122.00   
mean        148.48         36922.53              213.36             149.29   
std          96.65         39371.17               97.67              88.10   
min          36.00          2212.00               61.44              31.00   
25%          76.25         12115.75              142.96              77.75   
50%         118.50         20266.00              192.18             127.00   
75%         205.50         44815.75              272.83             212.25   
max         468.00        181189.00              473.06             383.50   

       pct_publico  
count       122.00  
mean          0.90  
std           0.13  
min           0.39  
25%           0.89  
50%           0.96  
75%           0.98  
max           1.00  


<h2 style="color:#A23B72; font-family:Georgia;">
📈 7. Tendencia del Abandono (2021→2025)
</h2>

Calculamos la diferencia entre el abandono más reciente y el más antiguo por distrito. Un valor positivo indica que el abandono **empeoró**.

In [7]:
ind_distrito = df_indicadores[df_indicadores["tipo"] == "distrito"][
    ["regional_distrito", año_col, "abandono_secundario"]
].copy()

# Pivotar abandono secundario por año
pivot = ind_distrito.pivot_table(
    index="regional_distrito", columns=año_col, values="abandono_secundario"
)

# Tendencia: último año - primer año
pivot["tendencia_abandono_sec"] = pivot["2024-2025"] - pivot["2021-2022"]
# Abandono promedio histórico
pivot["abandono_sec_historico"] = pivot[["2021-2022", "2022-2023", "2023-2024", "2024-2025"]].mean(axis=1)

tendencia = pivot[["tendencia_abandono_sec", "abandono_sec_historico"]].round(2)
tendencia.index.name = "distrito"

df = df.merge(tendencia, on="distrito", how="left")

print("Tendencia del abandono secundario por distrito:")
print(tendencia.describe().round(2))
print(f"\nDistritos donde empeoró: {(tendencia['tendencia_abandono_sec'] > 0).sum()}")
print(f"Distritos donde mejoró: {(tendencia['tendencia_abandono_sec'] < 0).sum()}")

Tendencia del abandono secundario por distrito:
año_lectivo  tendencia_abandono_sec  abandono_sec_historico
count                        122.00                  122.00
mean                          -0.31                    5.63
std                            2.09                    1.70
min                           -6.00                    2.25
25%                           -1.50                    4.42
50%                            0.00                    5.38
75%                            1.20                    6.62
max                            4.90                   11.05

Distritos donde empeoró: 58
Distritos donde mejoró: 59


<h2 style="color:#A23B72; font-family:Georgia;">
⚠️ 8. Clasificación de Riesgo
</h2>

Asignamos una categoría de riesgo de deserción basada en la tasa de abandono máxima relevante del centro.

In [8]:
def clasificar_riesgo(abandono):
    if pd.isna(abandono):
        return "sin_datos"
    if abandono >= 8:
        return "alto"
    if abandono >= 5:
        return "medio"
    if abandono >= 3:
        return "bajo"
    return "muy_bajo"

df["riesgo"] = df["abandono_max"].apply(clasificar_riesgo)

print("Distribución de riesgo:")
print(df["riesgo"].value_counts())
print(f"\n% por categoría:")
print((df["riesgo"].value_counts(normalize=True) * 100).round(1))

Distribución de riesgo:
riesgo
muy_bajo     7468
medio        4118
bajo         3641
sin_datos    2306
alto          581
Name: count, dtype: int64

% por categoría:
riesgo
muy_bajo     41.2
medio        22.7
bajo         20.1
sin_datos    12.7
alto          3.2
Name: proportion, dtype: float64


<h2 style="color:#A23B72; font-family:Georgia;">
💾 9. Guardar Dataset Final
</h2>

In [9]:
df.to_csv(PROCESSED_DIR / "dataset_modelo.csv", index=False)

print(f"Dataset final guardado: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")

Dataset final guardado: (18114, 37)
Columnas: ['regional', 'distrito', 'centros', 'sector', 'nivel', 'coordenadas latitud', 'coordenadas longitud', 'matricula', 'planta fisica', 'provincia', 'municipio', 'año', 'abandono_inicial', 'promovido_inicial', 'reprobado_inicial', 'abandono_primario', 'promovido_primario', 'reprobado_primario', 'abandono_secundario', 'promovido_secundario', 'reprobado_secundario', 'tiene_inicial', 'tiene_primario', 'tiene_secundario', 'tiene_adultos', 'num_niveles', 'es_publico', 'abandono_max', 'abandono_promedio', 'num_centros', 'matricula_total', 'matricula_promedio', 'matricula_mediana', 'pct_publico', 'tendencia_abandono_sec', 'abandono_sec_historico', 'riesgo']


<h2 style="color:#A23B72; font-family:Georgia;">
🔍 10. Resumen del Dataset
</h2>

In [10]:
print("=" * 60)
print("RESUMEN DEL DATASET PARA MODELADO")
print("=" * 60)
print(f"\nFilas: {len(df)}")
print(f"Columnas: {len(df.columns)}")
print(f"\nNulos por columna:")
nulos = df.isnull().sum()
print(nulos[nulos > 0])
print(f"\nVariables originales: regional, distrito, centros, sector, nivel,")
print(f"  coordenadas, matricula, planta fisica, provincia, municipio, año")
print(f"\nFeatures creadas:")
print(f"  - tiene_inicial/primario/secundario/adultos (flags de nivel)")
print(f"  - num_niveles, es_publico")
print(f"  - abandono_max, abandono_promedio (tasa relevante por centro)")
print(f"  - num_centros, matricula_total/promedio/mediana (por distrito)")
print(f"  - pct_publico (por distrito)")
print(f"  - tendencia_abandono_sec, abandono_sec_historico")
print(f"  - riesgo (alto/medio/bajo/muy_bajo)")

df.info()

RESUMEN DEL DATASET PARA MODELADO

Filas: 18114
Columnas: 37

Nulos por columna:
coordenadas latitud     10368
coordenadas longitud      325
abandono_max             2306
dtype: int64

Variables originales: regional, distrito, centros, sector, nivel,
  coordenadas, matricula, planta fisica, provincia, municipio, año

Features creadas:
  - tiene_inicial/primario/secundario/adultos (flags de nivel)
  - num_niveles, es_publico
  - abandono_max, abandono_promedio (tasa relevante por centro)
  - num_centros, matricula_total/promedio/mediana (por distrito)
  - pct_publico (por distrito)
  - tendencia_abandono_sec, abandono_sec_historico
  - riesgo (alto/medio/bajo/muy_bajo)
<class 'pandas.DataFrame'>
RangeIndex: 18114 entries, 0 to 18113
Data columns (total 37 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   regional                18114 non-null  str    
 1   distrito                18114 non-null  str    
 2   centros 